# Stage 2: option-pricing basics

This notebook uses synthetic inputs from `config/options.yaml`. It does not download prices or make a claim about a current SPY option. Read [`docs/options-pricing.md`](../docs/options-pricing.md) for formulas, assumptions, and limitations.

Before running the comparison, predict whether the European binomial price will be above, below, or close to the Black-Scholes price. Then ask how that gap changes when you increase the tree steps.

## 1. Load configuration and understand the inputs

Spot and strike are prices. Time, rates, dividend yield, and volatility are annualized; rates and volatility are decimals (`0.20` means 20%). Days are converted using ACT/365. A price is not meaningful without its assumptions.

In [ ]:
from quant_models.options import binomial_tree_price, black_scholes_price
from quant_models.options.config import load_pricing_example_settings

settings = load_pricing_example_settings()
time_to_expiry = settings.days_to_expiry / 365
settings

## 2. Compare two European-option methods

Black-Scholes is the analytic reference under its assumptions. The CRR tree approximates the same European contract with discrete up/down moves. First use the configured step count, then change `steps` to compare convergence.

In [ ]:
inputs = {
    "spot": settings.spot,
    "strike": settings.strike,
    "time_to_expiry_years": time_to_expiry,
    "risk_free_rate": settings.risk_free_rate,
    "volatility": settings.volatility,
    "option_type": "call",
    "dividend_yield": settings.dividend_yield,
}
analytic = black_scholes_price(**inputs)
tree = binomial_tree_price(**inputs, steps=settings.tree_steps)
analytic, tree, tree - analytic

## 3. Early exercise is a different contract

An American option can be exercised before expiry. The binomial tree checks that choice at each node. Black-Scholes remains European-only, so compare like with like when testing convergence; use Black-Scholes only as a European reference or lower bound where appropriate.

In [ ]:
american_put = binomial_tree_price(
    **{**inputs, "option_type": "put"},
    steps=settings.tree_steps,
    exercise_style="american",
)
american_put

## What can mislead you?

- A converged numerical method can still price the wrong assumptions very precisely.
- Constant volatility misses the volatility smile and changing market conditions.
- A saved yfinance quote can be stale or unsynchronized; this notebook does not calibrate to it.
- American early exercise and European exercise are not interchangeable.